In [4]:
# Install the Python connection library
!pip install sqlalchemy pymysql pandas

In [6]:
import pandas as pd
from sqlalchemy import create_engine, URL

connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="Arul@nitha08",
    host="localhost",
    port=3306,
    database="olist_db"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    print("MySQL connected successfully!")

MySQL connected successfully!


In [7]:
# Cross-check the SQL data using Python

tables = [
    "customers",
    "geolocation",
    "sellers",
    "product_category_name_translation",
    "products",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews"
]

for table in tables:
    df = pd.read_sql(
        f"SELECT COUNT(*) AS row_count FROM `{table}`",
        engine
    )
    print(table, ":", df.loc[0, "row_count"])

customers : 99441
geolocation : 19010
sellers : 3095
product_category_name_translation : 71
products : 32951
orders : 99441
order_items : 112650
order_payments : 103877
order_reviews : 98669


In [8]:
# Check missing values in the orders table
orders = pd.read_sql("SELECT * FROM orders", engine)

print("Shape:", orders.shape)
print("\nMissing values:")
print(orders.isna().sum())

Shape: (99441, 15)

Missing values:
ID                                  0
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
date_issue                          0
delivery_days                    2965
estimated_days                      0
delay_days                       2965
is_late                             0
purchase_month                      0
dtype: int64


In [10]:
# Check duplicate order IDs
print("Duplicate order IDs:",
      orders["order_id"].duplicated().sum())

Duplicate order IDs: 0


In [11]:
# Check the relationship between orders and customers
query = """
SELECT COUNT(*) AS orphan_orders
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;
"""

result = pd.read_sql(query, engine)
print(result)

   orphan_orders
0              0
